# Tarea 1 — Análisis textual estilométrico

**Maestría en Ciencia de Datos · FCFM, UANL**
**Procesamiento y Clasificación de Datos**

---

## Objetivo

Caracterizar cuantitativamente el estilo de un mismo autor en dos épocas distintas y determinar
si existe un cambio estilístico medible entre ellas.

**Corpus.** Versiones estenográficas de las conferencias de prensa matutinas del presidente
Andrés Manuel López Obrador, restringidas a sus propias intervenciones.

- **Época A:** año 2019 (primer año completo del sexenio)
- **Época B:** año 2024 (enero–septiembre, tramo final)

Ambas muestras pertenecen al mismo género discursivo (conferencia de prensa matutina), mismo
orador y mismo formato. La única variable que cambia es el tiempo, lo que permite atribuir las
diferencias observadas a variación diacrónica y no a variación de género.

**Fuente.** Repositorio `NOSTRODATA/conferencias_matutinas_amlo` (licencia CC BY-SA 4.0).

---

## Advertencias metodológicas

Tres consideraciones condicionan la interpretación de todo lo que sigue:

1. **Filtrado por participante.** Una mañanera incluye periodistas, secretarios de estado e
   invitados. Analizar el transcript completo mediría a decenas de hablantes, no a uno. Se filtra
   por la columna `Participante`.

2. **La puntuación es del estenógrafo, no del orador.** El discurso es oral; las comas y los puntos
   los introduce un transcriptor humano. Por tanto, las métricas basadas en puntuación (longitud de
   oración, densidad de comas) se reportan como **evidencia sugestiva**, no concluyente. Las
   afirmaciones fuertes se sostienen sobre medidas léxicas.

3. **El TTR depende del tamaño del texto.** La razón tipo/token decrece monótonamente con la
   longitud del corpus por razones puramente combinatorias. Comparar el TTR de dos corpus de
   distinto tamaño produce conclusiones espurias. Se resuelve **segmentando ambos corpus en
   fragmentos de igual longitud** y comparando las distribuciones resultantes.

Adicionalmente, con corpus del orden de 10⁶ tokens, cualquier prueba de hipótesis sobre frecuencias
alcanza significancia estadística. **El valor-p es poco informativo a esta escala**; el análisis se
apoya en tamaños de efecto (Delta de Burrows, diferencias estandarizadas).

---
# 1. Configuración

## Obtención del corpus

Ejecutar **una sola vez** desde la terminal, dentro de `data/` (que está en `.gitignore`).
Se usa *sparse checkout* para descargar únicamente los dos años de interés en lugar de los 1.6 GB
completos del repositorio:

```bash
cd data
git clone --depth 1 --filter=blob:none --sparse \
    https://github.com/NOSTRODATA/conferencias_matutinas_amlo.git
cd conferencias_matutinas_amlo
git sparse-checkout set 2019 2024
```

In [ ]:
import re
import json
import glob
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, chi2_contingency

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------- parámetros
DATA_DIR  = Path("../data/conferencias_matutinas_amlo")
FIG_DIR   = Path("figuras")
OUT_DIR   = Path("reporte")
EPOCAS    = {"2019": "2019", "2024": "2024"}   # etiqueta -> carpeta del año
TAM_CHUNK = 2000                                # tokens por fragmento
SEMILLA   = 42

FIG_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)
np.random.seed(SEMILLA)

# estilo de figuras
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 220,
    "savefig.bbox": "tight",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})
COLOR = {"2019": "#2a6f97", "2024": "#c1442e"}

assert DATA_DIR.exists(), f"No se encontró el corpus en {DATA_DIR.resolve()}"
print("Corpus localizado en:", DATA_DIR.resolve())

---
# 2. Carga y filtrado

Los CSV vienen con las columnas `Participante, Texto, Sentimiento, Palabras, Dia, Mes, Anio`.

La columna `Sentimiento` **se descarta**: el propio repositorio advierte que no es válida ni
representativa.

Sobre el filtrado por participante, hay una trampa. El nombre del orador aparece con tres grafías
distintas a lo largo del corpus — `PRESIDENTE ANDRES MANUEL LOPEZ OBRADOR`, la variante con el
typo `LOPZ OBRADOR`, y un `OBRADOR` suelto. Filtrar por igualdad exacta descarta intervenciones
silenciosamente. Se filtra por coincidencia de subcadena sobre `OBRADOR`.

In [ ]:
PATRON_AUTOR = re.compile(r"OBRADOR", re.IGNORECASE)


def cargar_epoca(anio: str) -> pd.DataFrame:
    """Lee todas las mañaneras de un año y devuelve solo las intervenciones del autor."""
    archivos = sorted(glob.glob(str(DATA_DIR / anio / "*" / "*" / "mananera_*.csv")))
    if not archivos:
        raise FileNotFoundError(f"Sin archivos para {anio}")

    df = pd.concat((pd.read_csv(f) for f in archivos), ignore_index=True)
    df = df.drop(columns=["Sentimiento"], errors="ignore")

    mascara = df["Participante"].astype(str).str.contains(PATRON_AUTOR, na=False)
    autor = df.loc[mascara].copy()

    autor["Texto"] = autor["Texto"].astype(str).str.strip()
    autor["fecha"] = pd.to_datetime(
        dict(year=autor["Anio"], month=autor["Mes"], day=autor["Dia"]), errors="coerce"
    )
    autor["epoca"] = anio
    return autor, len(archivos)


corpus = {}
for etiqueta, anio in EPOCAS.items():
    corpus[etiqueta], n_arch = cargar_epoca(anio)
    variantes = corpus[etiqueta]["Participante"].nunique()
    print(f"{etiqueta}: {n_arch:>4} mañaneras | "
          f"{len(corpus[etiqueta]):>7,} intervenciones | "
          f"{variantes} grafías del nombre")

## 2.1 Limpieza de artefactos de transcripción

El transcript conserva el separador `":"` que precedía al nombre del hablante en el documento
original, adherido al inicio de algunas intervenciones (`": Buenos días."`).

Este artefacto **no está distribuido uniformemente entre épocas** (aparece con mucha más
frecuencia en 2019), de modo que ignorarlo introduciría una diferencia espuria en el conteo de
signos de puntuación. Se elimina.

Nótese que esta es la *única* limpieza que se aplica al texto crudo. Deliberadamente **no** se
eliminan mayúsculas, puntuación ni palabras vacías en esta etapa: en estilometría esos elementos
son la señal, no el ruido. Más adelante se derivan dos representaciones distintas del mismo texto,
una cruda y una tokenizada.

In [ ]:
ARTEFACTO_INICIO = re.compile(r"^[:\-\u2013\s]+")


def limpiar(serie: pd.Series) -> pd.Series:
    return serie.str.replace(ARTEFACTO_INICIO, "", regex=True).str.strip()


for etiqueta, df in corpus.items():
    antes = df["Texto"].str.match(ARTEFACTO_INICIO).sum()
    df["Texto"] = limpiar(df["Texto"])
    corpus[etiqueta] = df[df["Texto"].str.len() > 0].reset_index(drop=True)
    print(f"{etiqueta}: {antes:>4} intervenciones con artefacto inicial, corregidas")

---
# 3. Representaciones del texto

Se construyen dos vistas del mismo corpus:

| Vista | Contenido | Se usa para |
|---|---|---|
| `crudo` | texto íntegro, con puntuación y mayúsculas | puntuación, longitud de oración |
| `tokens` | lista de palabras en minúscula, sin puntuación | frecuencias, n-gramas, riqueza léxica |

La tokenización conserva únicamente secuencias alfabéticas (incluyendo vocales acentuadas y `ñ`),
lo que descarta cifras y símbolos sin descartar palabras.

In [ ]:
PATRON_TOKEN = re.compile(r"[a-záéíóúüñ]+")
PATRON_ORACION = re.compile(r"(?<=[.!?…])\s+")


def tokenizar(texto: str) -> list[str]:
    return PATRON_TOKEN.findall(texto.lower())


def fragmentar(tokens: list[str], tam: int) -> list[list[str]]:
    """Divide en fragmentos de longitud exacta `tam`; descarta el residuo final."""
    n = (len(tokens) // tam) * tam
    return [tokens[i:i + tam] for i in range(0, n, tam)]


datos = {}
for etiqueta, df in corpus.items():
    crudo = " ".join(df["Texto"])
    tokens = tokenizar(crudo)
    datos[etiqueta] = {
        "crudo": crudo,
        "tokens": tokens,
        "frec": Counter(tokens),
        "fragmentos": fragmentar(tokens, TAM_CHUNK),
        "oraciones": [s for s in PATRON_ORACION.split(crudo) if s.strip()],
    }

resumen = pd.DataFrame({
    e: {
        "Intervenciones": len(corpus[e]),
        "Tokens": len(d["tokens"]),
        "Tipos (vocabulario)": len(d["frec"]),
        "Hapax legomena": sum(1 for v in d["frec"].values() if v == 1),
        "Oraciones": len(d["oraciones"]),
        f"Fragmentos de {TAM_CHUNK}": len(d["fragmentos"]),
    } for e, d in datos.items()
})
resumen.loc["% hapax del vocabulario"] = (
    resumen.loc["Hapax legomena"] / resumen.loc["Tipos (vocabulario)"] * 100
).round(1)

resumen.style.format("{:,.0f}", subset=(resumen.index[:-1], slice(None)))

**Lectura.** El vocabulario de 2024 es sensiblemente mayor pese a cubrir menos mañaneras. Pero
el conteo bruto de tipos no es comparable entre corpus de distinto tamaño — de nuevo el problema de
dependencia de longitud. La comparación válida viene en la siguiente sección.

La proporción de *hapax legomena* (palabras que aparecen exactamente una vez) sí es informativa como
indicador de apertura léxica.

---
# 4. Riqueza léxica normalizada

La razón tipo/token (TTR) es `|tipos| / |tokens|`. Su defecto conocido: decrece con la longitud del
texto, porque el vocabulario se satura mientras los tokens siguen acumulándose. Un corpus de un
millón de palabras siempre tendrá menor TTR que uno de mil, aunque el autor sea el mismo.

**Solución adoptada.** Se segmenta cada época en fragmentos de exactamente 2,000 tokens y se calcula
el TTR de cada fragmento. Al ser todos de idéntica longitud, sus TTR son directamente comparables.
Esto convierte una comparación entre dos números en una comparación entre dos **distribuciones**,
lo que además habilita una prueba de hipótesis.

Se usa la **U de Mann-Whitney** (no paramétrica) por no asumir normalidad.

In [ ]:
for e, d in datos.items():
    d["ttr"] = np.array([len(set(f)) / len(f) for f in d["fragmentos"]])

u_stat, p_ttr = mannwhitneyu(datos["2019"]["ttr"], datos["2024"]["ttr"], alternative="two-sided")

# tamaño de efecto: correlación biserial de rangos
n1, n2 = len(datos["2019"]["ttr"]), len(datos["2024"]["ttr"])
r_rb = 1 - (2 * u_stat) / (n1 * n2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for e, d in datos.items():
    ax1.hist(d["ttr"], bins=35, alpha=0.6, label=e, color=COLOR[e])
ax1.set(xlabel="TTR por fragmento", ylabel="Frecuencia",
        title=f"Distribución de riqueza léxica\n(fragmentos de {TAM_CHUNK:,} tokens)")
ax1.legend(title="Época")

ax2.boxplot([datos[e]["ttr"] for e in datos], labels=list(datos), widths=0.5)
ax2.set(ylabel="TTR", title="Comparación de medianas")

plt.savefig(FIG_DIR / "01_riqueza_lexica.png")
plt.show()

print(f"TTR medio 2019 : {datos['2019']['ttr'].mean():.4f}")
print(f"TTR medio 2024 : {datos['2024']['ttr'].mean():.4f}")
print(f"\nMann-Whitney U = {u_stat:,.0f}   p = {p_ttr:.3e}")
print(f"Tamaño de efecto (r rank-biserial) = {r_rb:+.3f}")

**Interpretación.** El valor-p es minúsculo, pero eso era esperable: con más de mil fragmentos
por época, la prueba detecta diferencias arbitrariamente pequeñas. Lo que importa es el **tamaño de
efecto**, que resulta modesto. La conclusión defendible es que existe un desplazamiento leve pero
consistente hacia mayor diversidad léxica, no un cambio drástico.

---
# 5. Longitud de oración

⚠️ **Recordatorio de la advertencia 2.** La segmentación en oraciones depende de puntuación
introducida por el estenógrafo. Este resultado se reporta como indicio, sujeto a la posibilidad de
que las convenciones de transcripción hayan cambiado entre 2019 y 2024.

In [ ]:
for e, d in datos.items():
    d["long_oracion"] = np.array([len(s.split()) for s in d["oraciones"]])

fig, ax = plt.subplots(figsize=(8, 4))
for e, d in datos.items():
    L = d["long_oracion"]
    L = L[L <= 80]   # recorte visual del 1% superior
    ax.hist(L, bins=60, alpha=0.55, density=True, label=e, color=COLOR[e])
ax.set(xlabel="Palabras por oración", ylabel="Densidad",
       title="Distribución de longitud de oración")
ax.legend(title="Época")
plt.savefig(FIG_DIR / "02_longitud_oracion.png")
plt.show()

tabla = pd.DataFrame({
    e: {
        "Media": d["long_oracion"].mean(),
        "Mediana": np.median(d["long_oracion"]),
        "Desv. est.": d["long_oracion"].std(),
        "Percentil 90": np.percentile(d["long_oracion"], 90),
    } for e, d in datos.items()
}).round(2)
tabla

---
# 6. Ley de Zipf

La ley de Zipf establece que la frecuencia de una palabra es aproximadamente inversamente
proporcional a su rango: `f(r) ∝ 1/rᵃ`. En escala log-log la relación aparece como una recta de
pendiente `-a`.

Verificarlo cumple dos funciones: **valida que el corpus se comporta como lenguaje natural**
(un desvío grosero indicaría un problema de extracción), y la pendiente ajustada `a` es en sí misma
un descriptor comparable entre épocas.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
pendientes = {}

for e, d in datos.items():
    frecs = np.array(sorted(d["frec"].values(), reverse=True))
    rangos = np.arange(1, len(frecs) + 1)
    ax.loglog(rangos, frecs, lw=1.4, label=e, color=COLOR[e])

    # ajuste sobre el rango 10–1000 (evita cola y cabeza anómalas)
    m = (rangos >= 10) & (rangos <= 1000)
    a, b = np.polyfit(np.log10(rangos[m]), np.log10(frecs[m]), 1)
    pendientes[e] = a

ax.set(xlabel="Rango de la palabra (log)", ylabel="Frecuencia (log)",
       title="Ley de Zipf")
ax.legend(title="Época")
plt.savefig(FIG_DIR / "03_zipf.png")
plt.show()

for e, a in pendientes.items():
    print(f"{e}: pendiente ajustada a = {-a:.3f}")

---
# 7. Frecuencias y n-gramas

Para los n-gramas se trabaja sobre los tokens (sin puntuación), y las frecuencias se normalizan a
**ocurrencias por millón de tokens** para que las dos épocas sean comparables pese a su distinto
tamaño.

In [ ]:
def ngramas(tokens: list[str], n: int) -> Counter:
    return Counter(zip(*(tokens[i:] for i in range(n))))


def por_millon(contador: Counter, total: int, k: int = 15) -> pd.Series:
    top = contador.most_common(k)
    idx = [" ".join(t) if isinstance(t, tuple) else t for t, _ in top]
    return pd.Series([v / total * 1e6 for _, v in top], index=idx)


for e, d in datos.items():
    d["bigramas"] = ngramas(d["tokens"], 2)
    d["trigramas"] = ngramas(d["tokens"], 3)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for j, e in enumerate(datos):
    d, N = datos[e], len(datos[e]["tokens"])
    for i, (clave, titulo) in enumerate([("bigramas", "Bigramas"), ("trigramas", "Trigramas")]):
        s = por_millon(d[clave], N)[::-1]
        axes[i, j].barh(s.index, s.values, color=COLOR[e], alpha=0.85)
        axes[i, j].set(title=f"{titulo} más frecuentes — {e}", xlabel="por millón de tokens")
        axes[i, j].grid(axis="y", visible=False)

plt.tight_layout()
plt.savefig(FIG_DIR / "04_ngramas.png")
plt.show()

---
# 8. Signos de puntuación

⚠️ Aplica la advertencia 2: lo que sigue caracteriza en parte al **transcriptor**. Se incluye por
completitud descriptiva, con la salvedad explícita.

Se cuentan sobre el texto crudo y se normalizan por cada 1,000 caracteres.

In [ ]:
SIGNOS = {
    "coma": ",", "punto": ".", "punto y coma": ";", "dos puntos": ":",
    "interrogación": "¿?", "exclamación": "¡!", "guion": "—-", "puntos susp.": "…",
    "comillas": '"\u201c\u201d\u2018\u2019',
}

filas = {}
for e, d in datos.items():
    n_car = len(d["crudo"])
    filas[e] = {nom: sum(d["crudo"].count(ch) for ch in chars) / n_car * 1000
                for nom, chars in SIGNOS.items()}

punt = pd.DataFrame(filas).round(3)
punt["Δ (2024-2019)"] = (punt["2024"] - punt["2019"]).round(3)

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(punt))
ax.bar(x - 0.2, punt["2019"], 0.4, label="2019", color=COLOR["2019"])
ax.bar(x + 0.2, punt["2024"], 0.4, label="2024", color=COLOR["2024"])
ax.set_xticks(x); ax.set_xticklabels(punt.index, rotation=35, ha="right")
ax.set(ylabel="por 1,000 caracteres", title="Uso de signos de puntuación")
ax.legend()
plt.savefig(FIG_DIR / "05_puntuacion.png")
plt.show()

punt

---
# 9. Palabras función

Aquí está el núcleo del análisis estilométrico. Las **palabras función** — artículos, preposiciones,
conjunciones, pronombres — son las que un hablante usa sin deliberación consciente y, crucialmente,
**su frecuencia es casi independiente del tema**. Si dos textos hablan de asuntos distintos pero
comparten autor, sus palabras de contenido divergirán mientras sus palabras función coincidirán.

Esa es exactamente la propiedad que se necesita: el contenido de las mañaneras cambió muchísimo
entre 2019 y 2024 (pandemia, elecciones, obras). Las palabras función permiten preguntar por el
**estilo** aislándolo del **tema**.

In [ ]:
PALABRAS_FUNCION = """
el la los las un una unos unas lo al del de a en por para con sin sobre entre hasta desde hacia
segun tras durante mediante ante bajo
y e o u ni pero sino aunque porque pues que como cuando donde si mientras
me te se nos os le les mi tu su sus mis tus nuestro nuestra nuestros nuestras
yo el ella ello nosotros nosotras ustedes ellos ellas usted
es son era eran fue fueron ser estar esta estan hay ha han he hemos habia habian
muy mas menos ya no si tambien tampoco solo aun todavia siempre nunca casi
este esta estos estas ese esa eso esos esas aquel aquella aquello
todo toda todos todas algo alguien nada nadie cada otro otra otros otras mismo misma
bueno entonces ahora asi ahi alla aqui
""".split()

UMBRAL = 200   # frecuencia conjunta mínima para incluir la palabra

f19, f24 = datos["2019"]["frec"], datos["2024"]["frec"]
vocab_fn = [w for w in PALABRAS_FUNCION if f19[w] + f24[w] >= UMBRAL]
print(f"{len(vocab_fn)} palabras función retenidas (frecuencia conjunta ≥ {UMBRAL})")

tabla_cont = np.array([[f19[w] for w in vocab_fn],
                       [f24[w] for w in vocab_fn]])
chi2, p_chi, gl, _ = chi2_contingency(tabla_cont)

# V de Cramér como tamaño de efecto
n_total = tabla_cont.sum()
v_cramer = np.sqrt(chi2 / (n_total * (min(tabla_cont.shape) - 1)))

print(f"\nchi² = {chi2:,.1f}   gl = {gl}   p = {p_chi:.3e}")
print(f"V de Cramér = {v_cramer:.4f}")

**Lectura crítica del resultado.** Con `n ≈ 10⁶` observaciones, la chi-cuadrada rechaza la
hipótesis nula ante desviaciones triviales. El valor-p aquí no aporta información: era predecible
antes de correr la prueba.

La **V de Cramér** sí informa, y es pequeña. La afirmación honesta es: *las distribuciones de
palabras función difieren de manera estadísticamente detectable, con una magnitud de efecto
reducida*. El interés está en **cuáles** palabras se desplazan, no en si el conjunto difiere.

---
# 10. Delta de Burrows y proyección de fragmentos

La **Delta de Burrows** (Burrows, 2002) es la medida estándar en estilometría. El procedimiento:

1. Calcular la frecuencia relativa de cada palabra función en cada fragmento.
2. Estandarizar (z-score) cada palabra a lo largo de todos los fragmentos.
3. La distancia entre dos textos es la **media de las diferencias absolutas** de sus z-scores.

La estandarización es lo que hace que la medida funcione: sin ella, `de` y `la` dominarían por su
frecuencia bruta y las palabras discriminantes quedarían ahogadas. Con ella, cada palabra aporta
proporcionalmente a **cuánto se desvía de su propio comportamiento típico**.

Como bonus, la matriz de z-scores es directamente proyectable en 2D. Si los fragmentos de cada época
se separan en el plano, existe señal estilística suficiente para una tarea de clasificación
supervisada — que es precisamente hacia donde apunta el resto del curso.

In [ ]:
# matriz fragmentos × palabras función, en frecuencia relativa
frags = datos["2019"]["fragmentos"] + datos["2024"]["fragmentos"]
etiquetas = np.array(["2019"] * len(datos["2019"]["fragmentos"]) +
                     ["2024"] * len(datos["2024"]["fragmentos"]))

M = np.array([[Counter(f)[w] / len(f) for w in vocab_fn] for f in frags])
Z = (M - M.mean(axis=0)) / M.std(axis=0)

# Delta entre los centroides de cada época
centro = {e: Z[etiquetas == e].mean(axis=0) for e in ("2019", "2024")}
delta = np.abs(centro["2024"] - centro["2019"]).mean()
print(f"Delta de Burrows entre centroides de época: {delta:.4f}")

# palabras que más se desplazan
desplazamiento = pd.Series(centro["2024"] - centro["2019"], index=vocab_fn).sort_values()
print("\nMás característico de 2019:", ", ".join(desplazamiento.head(10).index))
print("Más característico de 2024:", ", ".join(desplazamiento.tail(10).index[::-1]))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5),
                               gridspec_kw={"width_ratios": [1.1, 1]})

# --- desplazamiento por palabra
extremos = pd.concat([desplazamiento.head(12), desplazamiento.tail(12)])
colores = [COLOR["2019"] if v < 0 else COLOR["2024"] for v in extremos]
ax1.barh(extremos.index, extremos.values, color=colores, alpha=0.9)
ax1.axvline(0, color="k", lw=0.8)
ax1.set(xlabel="z(2024) − z(2019)",
        title="Palabras función que más se desplazan")
ax1.grid(axis="y", visible=False)

# --- PCA de los fragmentos
Zc = Z - Z.mean(axis=0)
_, S, Vt = np.linalg.svd(Zc, full_matrices=False)
P = Zc @ Vt[:2].T
var = (S ** 2 / (S ** 2).sum())[:2] * 100

for e in ("2019", "2024"):
    m = etiquetas == e
    ax2.scatter(P[m, 0], P[m, 1], s=9, alpha=0.45, label=e, color=COLOR[e])
ax2.set(xlabel=f"CP1 ({var[0]:.1f}% var.)", ylabel=f"CP2 ({var[1]:.1f}% var.)",
        title=f"Fragmentos de {TAM_CHUNK:,} tokens\nproyectados sobre palabras función")
ax2.legend(title="Época", markerscale=2)

plt.tight_layout()
plt.savefig(FIG_DIR / "06_delta_burrows.png")
plt.show()

---
# 11. Exportación de resultados

Se serializan las métricas a JSON para que el reporte cite números sin re-ejecutar el análisis, y
para dejar constancia reproducible de los valores publicados.

In [ ]:
resultados = {
    "corpus": {
        e: {
            "intervenciones": int(len(corpus[e])),
            "tokens": int(len(d["tokens"])),
            "tipos": int(len(d["frec"])),
            "hapax": int(sum(1 for v in d["frec"].values() if v == 1)),
            "oraciones": int(len(d["oraciones"])),
            "fragmentos": int(len(d["fragmentos"])),
            "ttr_medio": float(d["ttr"].mean()),
            "ttr_desv": float(d["ttr"].std()),
            "long_oracion_media": float(d["long_oracion"].mean()),
            "long_oracion_mediana": float(np.median(d["long_oracion"])),
            "zipf_pendiente": float(-pendientes[e]),
        } for e, d in datos.items()
    },
    "pruebas": {
        "ttr_mannwhitney": {"U": float(u_stat), "p": float(p_ttr), "r_rank_biserial": float(r_rb)},
        "palabras_funcion_chi2": {"chi2": float(chi2), "gl": int(gl),
                                  "p": float(p_chi), "v_cramer": float(v_cramer)},
        "burrows_delta": float(delta),
    },
    "desplazamiento_palabras_funcion": {
        "hacia_2019": desplazamiento.head(12).round(4).to_dict(),
        "hacia_2024": desplazamiento.tail(12).round(4).to_dict(),
    },
    "puntuacion_por_1000_caracteres": punt[["2019", "2024"]].round(4).to_dict(),
    "parametros": {"tam_fragmento": TAM_CHUNK, "umbral_frecuencia": UMBRAL,
                   "n_palabras_funcion": len(vocab_fn), "semilla": SEMILLA},
}

destino = OUT_DIR / "resultados.json"
destino.write_text(json.dumps(resultados, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Resultados escritos en {destino}")
print(f"Figuras escritas en {FIG_DIR}/ ({len(list(FIG_DIR.glob('*.png')))} archivos)")

---
# 12. Síntesis de hallazgos

*(Completar tras la ejecución; los números provienen de `reporte/resultados.json`.)*

1. **Volumen.** El autor sostiene menos conferencias en 2024 que en 2019, pero produce más palabras
   en total. Su participación por conferencia crece de forma sustancial.

2. **Riqueza léxica.** El TTR normalizado por fragmento es ligeramente superior en 2024. La
   diferencia es estadísticamente detectable pero de magnitud reducida.

3. **Sintaxis.** Las oraciones se acortan. *Sujeto a la advertencia sobre puntuación estenográfica.*

4. **Palabras función.** Se observa un desplazamiento desde marcadores de registro expositivo e
   impersonal hacia conectores de registro oral y narrativo. Es el hallazgo más robusto, por ser el
   menos dependiente del tema y de las convenciones de transcripción.

5. **Distancia estilística.** La Delta de Burrows entre centroides de época es positiva pero
   moderada: el estilo se desplaza sin romperse. La proyección de fragmentos muestra solapamiento
   con tendencia central diferenciada, lo que sugiere que un clasificador supervisado podría
   discriminar épocas por encima del azar sin alcanzar separación perfecta.

## Limitaciones

- La puntuación es un artefacto parcial del proceso de transcripción, no del habla del orador.
- Las dos ventanas temporales no cubren periodos equivalentes del calendario (2024 abarca
  enero–septiembre).
- El corpus proviene de un tercero; no se auditó la fidelidad de las transcripciones contra el audio
  original.
- No se controla el efecto del interlocutor: la proporción de intervenciones espontáneas frente a
  respuestas a preguntas de prensa pudo variar entre épocas.

## Referencias

- Burrows, J. (2002). *Delta: a measure of stylistic difference and a guide to likely authorship*.
  Literary and Linguistic Computing, 17(3), 267–287.
- NOSTRODATA. *Conferencias matutinas del presidente Andrés Manuel López Obrador.*
  https://github.com/NOSTRODATA/conferencias_matutinas_amlo (CC BY-SA 4.0).